In [ ]:
import pandas as pd
import glob
import math
import string
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
import numpy as np

# Define the sheet index to load data from
# sheet = 0: sensitive cells + radiotherapy
# sheet = 1: resistant cells + radiotherapy
# sheet = 2: sensitive cells + radiotherapy + immunotherapy
for sheet_name in [0,1,2]:
    # Read the Excel file into a DataFrame
    df = pd.read_excel('Patrick_Data/Radiation+Immunotherapy_CD8_Tumor_Volume.xlsx', sheet_name=sheet_name)
    
    # Extract the first key (column name) as the initial details about the data
    details_about_data = df.keys()[0]
    
    # Set the first row as the column headers and remove it from the DataFrame
    df.columns = df.iloc[0]
    df = df.drop(0).reset_index(drop=True)
    
    # Concatenate additional information from the first column of the updated DataFrame
    details_about_data += " " + df.keys()[0]
    
    # Again, set the first row as the column headers and remove it from the DataFrame
    df.columns = df.iloc[0]
    df = df.drop(0).reset_index(drop=True)
    
    # Store the current column headers (treatment days) for later use
    treatment_days = df.keys()
    
    # One more time, set the first row as the column headers and remove it from the DataFrame
    df.columns = df.iloc[0]
    df = df.drop(0).reset_index(drop=True)
    
    # Store the current column headers (measurement days)
    measurement_days = df.keys()
    
    # Finally, set the first row as the column headers and drop it from the DataFrame
    df.columns = df.iloc[0]
    df = df.drop(0).reset_index(drop=True)
    
    # Display the final processed DataFrame
    display(df)
    
    # Extract the measurement days as integers from the column names where 'Day' is present
    measurement_days_int = np.array([int(col.split()[-1]) for col in measurement_days if 'Day' in str(col)])
    
    # Extract the treatment days as integers from the column names where 'Radiation' is mentioned
    treatment_days_int = np.array([int(measurement_days[i].split()[-1]) for i, col in enumerate(treatment_days) if 'Radiation' in str(col)])
    
    # Print the extracted measurement and treatment days
    print(f'Measurement days: {measurement_days_int}')
    print(f'Treatment days: {treatment_days_int}')
    
    # Print the number of rows (mice) in the DataFrame
    print(f'Number of mice: {df.shape[0]}')
    
    # Convert all column names to strings to avoid issues with non-string types like datetime
    df.columns = df.columns.map(str)
    
    # Find the column that contains the word 'Cohort' in its name
    # Assuming there's only one such column, and we select the first occurrence
    if sheet_name < 2:
        cohort_column = [col for col in df.columns if 'Cohort' in col]
        cohort_column = cohort_column[0]
    
    # Find the column that contains the word 'Mouse' in its name
    # Similarly, we assume there's only one such column
    mouse_column = [col for col in df.columns if 'Mouse' in col]
    mouse_column = mouse_column[0]
    
    # Find the column that contains the word 'Treatment' in its name
    # Again, selecting the first match
    if sheet_name < 2:
        treatment_column = [col for col in df.columns if 'Treatment' in col]
        treatment_column = treatment_column[0]
    else:
        treatment_column = df.keys()[1]
    
    # Find all columns that contain the word 'Volume' in their name
    # Again, selecting the first match
    volume_column = [col for col in df.columns if 'Volume' in col]
    volume_column = volume_column[0]
    
    # Iterate through each row in the DataFrame
    for index, row in df.iterrows():
        # Determine the cell type based on the sheet_name
        if sheet_name == 0 or sheet_name == 2:
            cell_type = 'sensitive'
        elif sheet_name == 1:
            cell_type = 'resistant'
        
        # Construct the base of the file name using treatment, cell type, cohort, and mouse information
        if sheet_name == 2:
            file_name = f"{row[treatment_column].iloc[0].replace(' ','').lower()}_{cell_type}_c0_m{row[mouse_column]}"
        else:
            file_name = f"{row[treatment_column].replace(' ','').lower()}_{cell_type}_c{row[cohort_column]}_m{row[mouse_column]}"
        
        # Append treatment days to the file name
        for t in treatment_days_int:
            file_name += '_t' + str(t)
        
        # Add the directory and file extension
        file_name = './data/' + file_name + '.txt'
        
        # Print the generated file name for verification
        print(file_name)
        
        # Extract volume measurements and convert them to float
        measurements = np.array([float(x) for x in row[volume_column].values])
        
        # Find the index of the first NaN value in the measurements array
        nan_index = np.where(np.isnan(measurements))[0]
        
        # If a NaN is found, truncate the array at the first occurrence of NaN
        if len(nan_index) > 0:
            measurements = measurements[:nan_index[0]]
        
        # Combine the measurement days and the truncated measurements into a 2D array
        # The number of measurements should match the number of measurement days
        data_to_save = np.column_stack((measurement_days_int[:len(measurements)], measurements))
        
        # Save the combined data (measurement days and volumes) to a text file
        np.savetxt(file_name, data_to_save)

In [ ]:
# Configure global settings for plot appearance
plt.rcParams["font.size"] = "14"  # Set font size for all text in plots

# Create a formatter for the y-axis that uses scientific notation when appropriate
formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-3, 2))  # Use scientific notation for numbers outside the range 10^-3 to 10^2

# Find and sort all text files in the ./data directory
data_files = sorted(glob.glob("./data/*.txt"))

# Dictionary to store the parsed data, grouped by data type
full_data = {}

# Loop through each file in the data_files list
for file in data_files:
    # Extract the data type (treatment and cell type) from the file name
    data_type = file.split('/')[-1].split('_c')[0]
    
    # Extract the mouse name (cohort identifier) from the file name
    mouse_name = 'c' + file.split('/')[-1].split('_c')[-1].split('_t')[0]
    
    # Extract the treatment days from the file name (as integers)
    t_days = np.array([int(t) for t in file.split('/')[-1].split('_c')[-1].split('.txt')[0].split('_t')[1:]])
    
    # Load the measurement data from the file (time, tumor volume)
    data = np.loadtxt(file)
    
    # Extract the specific treatment from the data type (e.g., radiation or control)
    treatment = data_type.split('_')[0]
    
    # If this data type hasn't been seen before, create a new list in full_data
    if data_type not in full_data:
        full_data[data_type] = []
    
    # Append the mouse's data (including name, tumor measurements, treatment, and treatment days) to the appropriate group
    full_data[data_type].append({
        'name': mouse_name,
        'data': data,
        'treatment': treatment,
        'treatment_days': t_days
    })

# Initialize a variable to keep track of the maximum tumor volume across all data
all_max = 0

# Loop through each group of data (e.g., sensitive, resistant)
for group in full_data:
    # Loop through each mouse in the group
    for mouse in full_data[group]:
        # Find the maximum tumor volume for this mouse
        m_max = max(mouse['data'][:, 1])  # column 1 holds the tumor volume
        # Update all_max if the current mouse's maximum is larger than the previous maximum
        if m_max > all_max:
            all_max = m_max

# Round up the maximum tumor volume to the nearest 1000 for nicer y-axis limits
rounded_max = math.ceil(all_max / 1000) * 1000

# Define the number of columns for the subplot grid (4 columns)
nCols = 4

# Loop through each group of data in full_data
for key in full_data:
    print(key)
    
    # Get the number of mice (scenarios) in the current group
    nScenarios = len(full_data[key])
    
    # Calculate the number of rows needed for the subplot grid
    nRows = int(np.ceil(nScenarios / nCols))
    
    # Create subplots with the calculated number of rows and columns, and set the figure size
    fig, axes = plt.subplots(nrows=nRows, ncols=nCols, figsize=(nCols * 7, nRows * 4))
    
    # Flatten the axes array to simplify indexing
    axes = axes.ravel()
    
    # Loop through each mouse in the current group and plot their data
    for i, m in enumerate(full_data[key]):
        ax = axes[i]
        # Label the plot with the mouse identifier, replacing 'M' with 'Mouse'
        ax.text(
            0.05,
            0.9,
            "{:}) {:}".format(string.ascii_uppercase[i], m['name']),
            horizontalalignment="left",
            transform=ax.transAxes,
            fontsize=18,
        )
        ax.set_xlabel("Time (days)")  # Set x-axis label
        ax.set_ylabel("Tumor Volume (mm$^3$)")  # Set y-axis label
        
        # Plot the tumor volume over time as a dashed line with markers
        ax.plot(m['data'][:, 0], m['data'][:, 1], "o", color="black", linestyle="dashed", linewidth=2, markersize=10)
        
        # Set the y-axis to use scientific notation when necessary
        ax.yaxis.set_major_formatter(formatter)
        
        # Set y-axis limits to start from 0 and go up to the rounded maximum value
        ax.set_ylim((0, rounded_max))
        
        # Set x-axis limits to represent 0 to 60 days
        ax.set_xlim((0, 60))
    
    # Remove any unused axes (if there are fewer mice than subplot slots)
    for i in range(nScenarios, nRows * nCols):
        fig.delaxes(axes[i])
    
    # Save the figure as a PDF file, naming it after the group (without spaces and in lowercase)
    plt.savefig("./figures/all_data_" + key + ".pdf", bbox_inches="tight", pad_inches=0.02)
    
    # Display the figure on the screen
    plt.show()

# Define the number of columns for the subplot grid (4 columns)
nCols = 3

# Get the number of mice (scenarios) in the current group
nScenarios = len(full_data)

# Calculate the number of rows needed for the subplot grid
nRows = int(np.ceil(nScenarios / nCols))

# Create subplots with the calculated number of rows and columns, and set the figure size
fig, axes = plt.subplots(nrows=nRows, ncols=nCols, figsize=(nCols * 7, nRows * 4))

# Flatten the axes array to simplify indexing
axes = axes.ravel()

# Loop through each group of data in full_data
for i, key in enumerate(full_data):
    ax = axes[i]
    # Label the plot with the group identifier
    ax.text(
        0.05,
        0.9,
        "{:}) {:}".format(string.ascii_uppercase[i], key.replace('_',' ').capitalize() + ', ' + str(len(full_data[key])) + ' mice'),
        horizontalalignment="left",
        transform=ax.transAxes,
        fontsize=18,
    )
    ax.set_xlabel("Time (days)")  # Set x-axis label
    ax.set_ylabel("Tumor Volume (mm$^3$)")  # Set y-axis label
    
    # Loop through each mouse in the current group and plot their data
    for j, m in enumerate(full_data[key]):

        # Plot the tumor volume over time as a dashed line with markers
        ax.plot(m['data'][:, 0], m['data'][:, 1], "o", linestyle="dashed", linewidth=2, markersize=10)
        
    # Set the y-axis to use scientific notation when necessary
    ax.yaxis.set_major_formatter(formatter)
    
    # Set y-axis limits to start from 0 and go up to the rounded maximum value
    ax.set_ylim((0, rounded_max))
    
    # Set x-axis limits to represent 0 to 60 days
    ax.set_xlim((0, 60))
    
# Remove any unused axes (if there are fewer mice than subplot slots)
for i in range(nScenarios, nRows * nCols):
    fig.delaxes(axes[i])

# Save the figure as a PDF file
plt.savefig("./figures/all_data.pdf", bbox_inches="tight", pad_inches=0.02)

# Display the figure on the screen
plt.show()

In [ ]:
# Configure global settings for plot appearance
plt.rcParams["font.size"] = "14"  # Set font size for all text in plots

# Create a formatter for the y-axis that uses scientific notation when appropriate
formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-3, 2))  # Use scientific notation for numbers outside the range 10^-3 to 10^2

# Find and sort all text files in the ./data directory
#data_files = sorted(glob.glob("./data/*.txt"))
data_files = sorted([file for file in glob.glob("./data/*.txt") if 'c0' not in file])

# Dictionary to store the parsed data, grouped by data type
full_data = {}

# Loop through each file in the data_files list
for file in data_files:
    # Extract the data type (treatment and cell type) from the file name
    data_type = file.split('/')[-1].split('_c')[0]
    
    # Extract the mouse name (cohort identifier) from the file name
    mouse_name = 'c' + file.split('/')[-1].split('_c')[-1].split('_t')[0]
    
    # Extract the treatment days from the file name (as integers)
    t_days = np.array([int(t) for t in file.split('/')[-1].split('_c')[-1].split('.txt')[0].split('_t')[1:]])
    
    # Load the measurement data from the file (time, tumor volume)
    data = np.loadtxt(file)
    
    # Extract the specific treatment from the data type (e.g., radiation or control)
    treatment = data_type.split('_')[0]
    
    # If this data type hasn't been seen before, create a new list in full_data
    if data_type not in full_data:
        full_data[data_type] = []
    
    # Append the mouse's data (including name, tumor measurements, treatment, and treatment days) to the appropriate group
    full_data[data_type].append({
        'name': mouse_name,
        'data': data,
        'treatment': treatment,
        'treatment_days': t_days
    })

# # Initialize a variable to keep track of the maximum tumor volume across all data
# all_max = 0

# # Loop through each group of data (e.g., sensitive, resistant)
# for group in full_data:
#     # Loop through each mouse in the group
#     for mouse in full_data[group]:
#         # Find the maximum tumor volume for this mouse
#         m_max = max(mouse['data'][:, 1])  # column 1 holds the tumor volume
#         # Update all_max if the current mouse's maximum is larger than the previous maximum
#         if m_max > all_max:
#             all_max = m_max

# # Round up the maximum tumor volume to the nearest 1000 for nicer y-axis limits
# rounded_max = math.ceil(all_max / 1000) * 1000

# Define the number of columns for the subplot grid (4 columns)
nCols = 4

# Loop through each group of data in full_data
for key in full_data:
    print(key)
    
    # Get the number of mice (scenarios) in the current group
    nScenarios = len(full_data[key])
    
    # Calculate the number of rows needed for the subplot grid
    nRows = int(np.ceil(nScenarios / nCols))
    
    # Create subplots with the calculated number of rows and columns, and set the figure size
    fig, axes = plt.subplots(nrows=nRows, ncols=nCols, figsize=(nCols * 7, nRows * 4))
    
    # Flatten the axes array to simplify indexing
    axes = axes.ravel()
    
    # Loop through each mouse in the current group and plot their data
    for i, m in enumerate(full_data[key]):
        ax = axes[i]
        # Label the plot with the mouse identifier, replacing 'M' with 'Mouse'
        ax.text(
            0.05,
            0.9,
            "{:}) {:}".format(string.ascii_uppercase[i], m['name']),
            horizontalalignment="left",
            transform=ax.transAxes,
            fontsize=18,
        )
        ax.set_xlabel("Time (days)")  # Set x-axis label
        ax.set_ylabel("Tumor Volume (mm$^3$)")  # Set y-axis label
        
        # Plot the tumor volume over time as a dashed line with markers
        ax.plot(m['data'][:, 0], m['data'][:, 1], "o", color="black", linestyle="dashed", linewidth=2, markersize=10)
        
        # Set the y-axis to use scientific notation when necessary
        ax.yaxis.set_major_formatter(formatter)
        
        # Set y-axis limits to start from 0 and go up to the rounded maximum value
        ax.set_ylim((0, rounded_max))
        
        # Set x-axis limits to represent 0 to 60 days
        ax.set_xlim((0, 60))
    
    # Remove any unused axes (if there are fewer mice than subplot slots)
    for i in range(nScenarios, nRows * nCols):
        fig.delaxes(axes[i])
    
    # Save the figure as a PDF file, naming it after the group (without spaces and in lowercase)
    plt.savefig("./figures/all_data_" + key + "_sheets01.pdf", bbox_inches="tight", pad_inches=0.02)
    
    # Display the figure on the screen
    plt.show()

# Define the number of columns for the subplot grid (4 columns)
nCols = 2

# Get the number of mice (scenarios) in the current group
nScenarios = len(full_data)

# Calculate the number of rows needed for the subplot grid
nRows = int(np.ceil(nScenarios / nCols))

# Create subplots with the calculated number of rows and columns, and set the figure size
fig, axes = plt.subplots(nrows=nRows, ncols=nCols, figsize=(nCols * 7, nRows * 4))

# Flatten the axes array to simplify indexing
axes = axes.ravel()

# Loop through each group of data in full_data
for i, key in enumerate(full_data):
    ax = axes[i]
    # Label the plot with the group identifier
    ax.text(
        0.05,
        0.9,
        "{:}) {:}".format(string.ascii_uppercase[i], key.replace('_',' ').capitalize() + ', ' + str(len(full_data[key])) + ' mice'),
        horizontalalignment="left",
        transform=ax.transAxes,
        fontsize=18,
    )
    ax.set_xlabel("Time (days)")  # Set x-axis label
    ax.set_ylabel("Tumor Volume (mm$^3$)")  # Set y-axis label
    
    # Loop through each mouse in the current group and plot their data
    for j, m in enumerate(full_data[key]):

        # Plot the tumor volume over time as a dashed line with markers
        ax.plot(m['data'][:, 0], m['data'][:, 1], "o", linestyle="dashed", linewidth=2, markersize=10)
        
    # Set the y-axis to use scientific notation when necessary
    ax.yaxis.set_major_formatter(formatter)
    
    # Set y-axis limits to start from 0 and go up to the rounded maximum value
    ax.set_ylim((0, rounded_max))
    
    # Set x-axis limits to represent 0 to 60 days
    ax.set_xlim((0, 60))
    
# Remove any unused axes (if there are fewer mice than subplot slots)
for i in range(nScenarios, nRows * nCols):
    fig.delaxes(axes[i])

# Save the figure as a PDF file
plt.savefig("./figures/all_data_sheets01.pdf", bbox_inches="tight", pad_inches=0.02)

# Display the figure on the screen
plt.show()

In [ ]:
# Configure global settings for plot appearance
plt.rcParams["font.size"] = "14"  # Set font size for all text in plots

# Create a formatter for the y-axis that uses scientific notation when appropriate
formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-3, 2))  # Use scientific notation for numbers outside the range 10^-3 to 10^2

# Find and sort all text files in the ./data directory
data_files = sorted([file for file in glob.glob("./data/*.txt") if 'c0' in file])

# Dictionary to store the parsed data, grouped by data type
full_data = {}

# Loop through each file in the data_files list
for file in data_files:
    # Extract the data type (treatment and cell type) from the file name
    data_type = file.split('/')[-1].split('_c')[0]
    
    # Extract the mouse name (cohort identifier) from the file name
    mouse_name = 'c' + file.split('/')[-1].split('_c')[-1].split('_t')[0]
    
    # Extract the treatment days from the file name (as integers)
    t_days = np.array([int(t) for t in file.split('/')[-1].split('_c')[-1].split('.txt')[0].split('_t')[1:]])
    
    # Load the measurement data from the file (time, tumor volume)
    data = np.loadtxt(file)
    
    # Extract the specific treatment from the data type (e.g., radiation or control)
    treatment = data_type.split('_')[0]
    
    # If this data type hasn't been seen before, create a new list in full_data
    if data_type not in full_data:
        full_data[data_type] = []
    
    # Append the mouse's data (including name, tumor measurements, treatment, and treatment days) to the appropriate group
    full_data[data_type].append({
        'name': mouse_name,
        'data': data,
        'treatment': treatment,
        'treatment_days': t_days
    })

# # Initialize a variable to keep track of the maximum tumor volume across all data
# all_max = 0

# # Loop through each group of data (e.g., sensitive, resistant)
# for group in full_data:
#     # Loop through each mouse in the group
#     for mouse in full_data[group]:
#         # Find the maximum tumor volume for this mouse
#         m_max = max(mouse['data'][:, 1])  # column 1 holds the tumor volume
#         # Update all_max if the current mouse's maximum is larger than the previous maximum
#         if m_max > all_max:
#             all_max = m_max

# # Round up the maximum tumor volume to the nearest 1000 for nicer y-axis limits
# rounded_max = math.ceil(all_max / 1000) * 1000

# Define the number of columns for the subplot grid (4 columns)
nCols = 4

# Loop through each group of data in full_data
for key in full_data:
    print(key)
    
    # Get the number of mice (scenarios) in the current group
    nScenarios = len(full_data[key])
    
    # Calculate the number of rows needed for the subplot grid
    nRows = int(np.ceil(nScenarios / nCols))
    
    # Create subplots with the calculated number of rows and columns, and set the figure size
    fig, axes = plt.subplots(nrows=nRows, ncols=nCols, figsize=(nCols * 7, nRows * 4))
    
    # Flatten the axes array to simplify indexing
    axes = axes.ravel()
    
    # Loop through each mouse in the current group and plot their data
    for i, m in enumerate(full_data[key]):
        ax = axes[i]
        # Label the plot with the mouse identifier, replacing 'M' with 'Mouse'
        ax.text(
            0.05,
            0.9,
            "{:}) {:}".format(string.ascii_uppercase[i], m['name']),
            horizontalalignment="left",
            transform=ax.transAxes,
            fontsize=18,
        )
        ax.set_xlabel("Time (days)")  # Set x-axis label
        ax.set_ylabel("Tumor Volume (mm$^3$)")  # Set y-axis label
        
        # Plot the tumor volume over time as a dashed line with markers
        ax.plot(m['data'][:, 0], m['data'][:, 1], "o", color="black", linestyle="dashed", linewidth=2, markersize=10)
        
        # Set the y-axis to use scientific notation when necessary
        ax.yaxis.set_major_formatter(formatter)
        
        # Set y-axis limits to start from 0 and go up to the rounded maximum value
        ax.set_ylim((0, rounded_max))
        
        # Set x-axis limits to represent 0 to 60 days
        ax.set_xlim((0, 60))
    
    # Remove any unused axes (if there are fewer mice than subplot slots)
    for i in range(nScenarios, nRows * nCols):
        fig.delaxes(axes[i])
    
    # Save the figure as a PDF file, naming it after the group (without spaces and in lowercase)
    plt.savefig("./figures/all_data_" + key + "_sheets2.pdf", bbox_inches="tight", pad_inches=0.02)
    
    # Display the figure on the screen
    plt.show()

# Define the number of columns for the subplot grid (4 columns)
nCols = 2

# Get the number of mice (scenarios) in the current group
nScenarios = len(full_data)

# Calculate the number of rows needed for the subplot grid
nRows = int(np.ceil(nScenarios / nCols))

# Create subplots with the calculated number of rows and columns, and set the figure size
fig, axes = plt.subplots(nrows=nRows, ncols=nCols, figsize=(nCols * 7, nRows * 4))

# Flatten the axes array to simplify indexing
axes = axes.ravel()

# Loop through each group of data in full_data
for i, key in enumerate(full_data):
    ax = axes[i]
    # Label the plot with the group identifier
    ax.text(
        0.05,
        0.9,
        "{:}) {:}".format(string.ascii_uppercase[i], key.replace('_',' ').capitalize() + ', ' + str(len(full_data[key])) + ' mice'),
        horizontalalignment="left",
        transform=ax.transAxes,
        fontsize=18,
    )
    ax.set_xlabel("Time (days)")  # Set x-axis label
    ax.set_ylabel("Tumor Volume (mm$^3$)")  # Set y-axis label
    
    # Loop through each mouse in the current group and plot their data
    for j, m in enumerate(full_data[key]):

        # Plot the tumor volume over time as a dashed line with markers
        ax.plot(m['data'][:, 0], m['data'][:, 1], "o", linestyle="dashed", linewidth=2, markersize=10)
        
    # Set the y-axis to use scientific notation when necessary
    ax.yaxis.set_major_formatter(formatter)
    
    # Set y-axis limits to start from 0 and go up to the rounded maximum value
    ax.set_ylim((0, rounded_max))
    
    # Set x-axis limits to represent 0 to 60 days
    ax.set_xlim((0, 60))
    
# Remove any unused axes (if there are fewer mice than subplot slots)
for i in range(nScenarios, nRows * nCols):
    fig.delaxes(axes[i])

# Save the figure as a PDF file
plt.savefig("./figures/all_data_sheets2.pdf", bbox_inches="tight", pad_inches=0.02)

# Display the figure on the screen
plt.show()